In [18]:
import requests
import pandas as pd
# 1. Получение информации обо всех книгах и сохранение в pd.DataFrame
def get_books():
    url = "https://www.anapioficeandfire.com/api/books"
    response = requests.get(url)
    if response.status_code == 200:
        books_data = response.json()
        return pd.DataFrame(books_data)
    else:
        print(f"Ошибка при получении данных о книгах: {response.status_code}")
        return pd.DataFrame()

books_df = get_books()
print("Книги:")
print(books_df.head())

Книги:
                                             url               name  \
0  https://www.anapioficeandfire.com/api/books/1  A Game of Thrones   
1  https://www.anapioficeandfire.com/api/books/2   A Clash of Kings   
2  https://www.anapioficeandfire.com/api/books/3  A Storm of Swords   
3  https://www.anapioficeandfire.com/api/books/4   The Hedge Knight   
4  https://www.anapioficeandfire.com/api/books/5  A Feast for Crows   

             isbn                authors  numberOfPages  \
0  978-0553103540  [George R. R. Martin]            694   
1  978-0553108033  [George R. R. Martin]            768   
2  978-0553106633  [George R. R. Martin]            992   
3  978-0976401100  [George R. R. Martin]            164   
4  978-0553801507  [George R. R. Martin]            784   

                   publisher        country     mediaType  \
0               Bantam Books  United States     Hardcover   
1               Bantam Books  United States      Hardback   
2               Bantam Books

In [19]:
# 2. Получение информации обо всех домах Вестероса и сохранение в pd.DataFrame
def get_houses():
    url = "https://www.anapioficeandfire.com/api/houses"
    response = requests.get(url)
    if response.status_code == 200:
        houses_data = response.json()
        return pd.DataFrame(houses_data)
    else:
        print(f"Ошибка при получении данных о домах: {response.status_code}")
        return pd.DataFrame()

houses_df = get_houses()
print("\nДома Вестероса:")
print(houses_df.head())


Дома Вестероса:
                                              url  \
0  https://www.anapioficeandfire.com/api/houses/1   
1  https://www.anapioficeandfire.com/api/houses/2   
2  https://www.anapioficeandfire.com/api/houses/3   
3  https://www.anapioficeandfire.com/api/houses/4   
4  https://www.anapioficeandfire.com/api/houses/5   

                          name           region  \
0                 House Algood  The Westerlands   
1  House Allyrion of Godsgrace            Dorne   
2                  House Amber        The North   
3                House Ambrose        The Reach   
4   House Appleton of Appleton        The Reach   

                                          coatOfArms            words titles  \
0  A golden wreath, on a blue field with a gold b...                      []   
1          Gyronny Gules and Sable, a hand couped Or  No Foe May Pass     []   
2                                                                         []   
3                             Or, sem

In [20]:
# 3. Получение информации обо всех домах Вестероса, у которых есть девиз
def get_houses_with_motto():
    url = "https://www.anapioficeandfire.com/api/houses?hasWords=true"
    response = requests.get(url)
    if response.status_code == 200:
        houses_data = response.json()
        return pd.DataFrame(houses_data)
    else:
        print(f"Ошибка при получении данных о домах с девизом: {response.status_code}")
        return pd.DataFrame()

houses_with_motto_df = get_houses_with_motto()
print("\nДома Вестероса с девизом:")
print(houses_with_motto_df.head())


Дома Вестероса с девизом:
                                               url  \
0   https://www.anapioficeandfire.com/api/houses/2   
1   https://www.anapioficeandfire.com/api/houses/4   
2   https://www.anapioficeandfire.com/api/houses/7   
3   https://www.anapioficeandfire.com/api/houses/8   
4  https://www.anapioficeandfire.com/api/houses/17   

                             name          region  \
0     House Allyrion of Godsgrace           Dorne   
1                   House Ambrose       The Reach   
2        House Arryn of the Eyrie        The Vale   
3        House Ashford of Ashford       The Reach   
4  House Baratheon of Storm's End  The Stormlands   

                                          coatOfArms                  words  \
0          Gyronny Gules and Sable, a hand couped Or        No Foe May Pass   
1                             Or, semy of ants gules          Never Resting   
2  A sky-blue falcon soaring against a white moon...       As High as Honor   
3  Tenny, a s

In [39]:
# 1. Подключение к БД
!pip install psycopg2-binary

import psycopg2
import requests
import pandas as pd

DB_HOSTNAME = 'hh-pgsql-public.ebi.ac.uk'
port = 5432
DB_DATABASE = 'pfmegrnargs'
DB_USER = 'reader'
DB_PASSWORD = 'NWDMCE5xdipIjRrp'

def connect_to_db():
    """Подключается к базе данных PostgreSQL и возвращает объект соединения."""
    try:
        conn = psycopg2.connect(
            host=DB_HOSTNAME,
            database=DB_DATABASE,
            user=DB_USER,
            password=DB_PASSWORD,
        )
        return conn
    except psycopg2.Error as e:
        print(f"Ошибка при подключении к базе данных: {e}")
        return None


In [40]:
# 2. Получение 10 строк из таблицы rnc_database (используем подход как в примере)
def get_rnc_data(conn):
    """Получает 10 строк из таблицы rnc_database и возвращает в DataFrame."""
    try:
        with conn.cursor() as cursor:
            cursor.execute("SELECT * FROM rnc_database LIMIT 10")
            rows = cursor.fetchall()

            # Получаем имена столбцов
            cursor.execute(
                "SELECT column_name FROM information_schema.columns WHERE table_name = 'rnc_database'"
            )
            columns = [col[0] for col in cursor.fetchall()]

            return pd.DataFrame(rows, columns=columns)
    except psycopg2.Error as e:  # Ловим psycopg2.Error
        print(f"Ошибка при выполнении запроса: {e}")
        return pd.DataFrame()



In [41]:
# 3. Получение значений столбцов display_name, num_sequences, num_organisms, url для 10 строк
def get_specific_columns(conn):
    """Получает значения определенных столбцов из таблицы rnc_database."""
    try:
        with conn.cursor() as cursor:  # Используем with для управления курсором
            cursor.execute(
                "SELECT display_name, num_sequences, num_organisms, url FROM rnc_database LIMIT 10;"
            )
            rows = cursor.fetchall()

            # Создаем DataFrame сразу здесь
            column_names = [
                "display_name",
                "num_sequences",
                "num_organisms",
                "url",
            ]  # Определяем имена столбцов
            df = pd.DataFrame(rows, columns=column_names)
            return df
    except psycopg2.Error as e:
        print(f"Ошибка при выполнении запроса: {e}")
        return None

In [42]:
# Выполнение кода

# API calls
books_df = get_books()
houses_df = get_houses()
houses_with_motto_df = get_houses_with_motto()

print("Книги:")
print(books_df.head())
print("\nВсе дома:")
print(houses_df.head())
print("\nДома с девизом:")
print(houses_with_motto_df.head())

# Database interaction
conn = connect_to_db()

if conn:
    # Получаем 10 строк из таблицы
    rnc_df = get_rnc_data(conn)
    print("\nПервые 10 строк из rnc_database:")
    print(rnc_df)

    # Получаем конкретные столбцы
    specific_columns_df = get_specific_columns(conn)
    print("\nКонкретные столбцы для 10 строк:")
    print(specific_columns_df)

    # Закрываем соединение
    conn.close()
else:
    print("Не удалось подключиться к базе данных")

Книги:
                                             url               name  \
0  https://www.anapioficeandfire.com/api/books/1  A Game of Thrones   
1  https://www.anapioficeandfire.com/api/books/2   A Clash of Kings   
2  https://www.anapioficeandfire.com/api/books/3  A Storm of Swords   
3  https://www.anapioficeandfire.com/api/books/4   The Hedge Knight   
4  https://www.anapioficeandfire.com/api/books/5  A Feast for Crows   

             isbn                authors  numberOfPages  \
0  978-0553103540  [George R. R. Martin]            694   
1  978-0553108033  [George R. R. Martin]            768   
2  978-0553106633  [George R. R. Martin]            992   
3  978-0976401100  [George R. R. Martin]            164   
4  978-0553801507  [George R. R. Martin]            784   

                   publisher        country     mediaType  \
0               Bantam Books  United States     Hardcover   
1               Bantam Books  United States      Hardback   
2               Bantam Books